In [113]:
from langchain_ollama import ChatOllama
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="llama3.1:latest", temperature=0)
db = SQLDatabase.from_uri("sqlite:///demo.db")
memory = InMemorySaver()

In [114]:
db.run("""
    CREATE TABLE IF NOT EXISTS tasks(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title VARCHAR(100) NOT NULL,
        description TEXT NOT NULL,
        status VARCHAR(30) DEFAULT 'pending',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
 """)

''

In [115]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [117]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(content="""
You are a SQL task assistant for a `tasks` table.

Rules:
- Use `ORDER BY created_at DESC LIMIT 10` for all task lists.
- After INSERT, UPDATE, or DELETE, run a SELECT to verify changes.
- If no records exist, reply: `No tasks found.`
- Return task lists as markdown tables.
- Keep responses short and precise.
- Prefer `id` over `title` when both exist.
- Do not generate unsupported SQL.

Schema:
tasks(
  id,
  title,
  description,
  status,
  created_at
)

Valid status values:
- pending
- in_progress
- completed

SQL Templates:

Create:
INSERT INTO tasks(title, description, status)
VALUES (?, ?, ?);

Read:
SELECT *
FROM tasks
WHERE ...
ORDER BY created_at DESC
LIMIT 10;

Update:
UPDATE tasks
SET status = ?
WHERE id = ? OR title = ?;

Delete:
DELETE FROM tasks
WHERE id = ? OR title = ?;
                              
Response Rules:
- Only display rows returned from the database.
- If rows exist, return a markdown table.
- If zero rows exist, not required to show the table format in this case and do not create mock data to display table records simply just return: "No tasks found"

Table format: (if records actually exists in the database)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
""")

In [118]:
system_prompt

SystemMessage(content='\nYou are a SQL task assistant for a `tasks` table.\n\nRules:\n- Use `ORDER BY created_at DESC LIMIT 10` for all task lists.\n- After INSERT, UPDATE, or DELETE, run a SELECT to verify changes.\n- If no records exist, reply: `No tasks found.`\n- Return task lists as markdown tables.\n- Keep responses short and precise.\n- Prefer `id` over `title` when both exist.\n- Do not generate unsupported SQL.\n\nSchema:\ntasks(\n  id,\n  title,\n  description,\n  status,\n  created_at\n)\n\nValid status values:\n- pending\n- in_progress\n- completed\n\nSQL Templates:\n\nCreate:\nINSERT INTO tasks(title, description, status)\nVALUES (?, ?, ?);\n\nRead:\nSELECT *\nFROM tasks\nWHERE ...\nORDER BY created_at DESC\nLIMIT 10;\n\nUpdate:\nUPDATE tasks\nSET status = ?\nWHERE id = ? OR title = ?;\n\nDelete:\nDELETE FROM tasks\nWHERE id = ? OR title = ?;\n\nResponse Rules:\n- Only display rows returned from the database.\n- If rows exist, return a markdown table.\n- If zero rows exist

In [119]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[]+tools,
    system_prompt=system_prompt,
    checkpointer=memory,
    debug=True
)

config = {"configurable": {"thread_id": "arik555"}}

In [120]:
message = {"messages": [
    HumanMessage(content="give me all the tasks")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='be19c080-42c7-4bdc-b472-b13e23707c4f')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:09:26.377828Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3376531708, 'load_duration': 82942250, 'prompt_eval_count': 763, 'prompt_eval_duration': 1938032791, 'eval_count': 29, 'eval_duration': 1350871124, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e2062-2338-72f3-b409-edea82fcf45b-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '242ccafe-8e5c-4619-bdc6-9a217da10c37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens': 792})]}}
[values] {'messages': [HumanMessage(content='giv

In [121]:
print(resp["messages"][-1].content)

No tasks found.


In [122]:
message = {"messages": [
    HumanMessage(content="Ok, create one task to study the RAG pipeline architecture")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='be19c080-42c7-4bdc-b472-b13e23707c4f'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:09:26.377828Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3376531708, 'load_duration': 82942250, 'prompt_eval_count': 763, 'prompt_eval_duration': 1938032791, 'eval_count': 29, 'eval_duration': 1350871124, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e2062-2338-72f3-b409-edea82fcf45b-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '242ccafe-8e5c-4619-bdc6-9a217da10c37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens': 792}), ToolMessage(content='', name='sql_db_query', id='0985c98c-7a41-4f91-8286-2f843da9ec4b

In [123]:
print(resp["messages"][-1].content)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
| 1  | Study RAG pipeline architecture | Learn about the RAG pipeline architecture | pending | 2026-05-13 08:11:43 |
